# SymbioPan v9 — Quickstart

A minimal end-to-end demo of the pipeline. Each step delegates to the official `scripts/` entry points.

## Workflow
1. **Preprocess** raw TIFF + GeoJSON → `.npy` patches in `dataset_processed/`.
2. **Train** Stage 1 model with rare-class weighted sampling.
3. **Infer** on held-out WSI tiles, with optional TTA.

All hyper-parameters live in `configs/defaults.py` (frozen dataclasses).

In [ ]:
import sys
import subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO), "-q"], check=True)
sys.path.insert(0, str(REPO))
print("SymbioPan installed.")

## 1. Inspect default configuration

In [ ]:
from configs import PATHS, STAGE1_DEFAULT_CONFIG, PREPROCESS_DEFAULT_CONFIG, INFERENCE_DEFAULT_CONFIG

print("Paths:")
for k, v in PATHS.__dict__.items():
    print(f"  {k}: {v}")
print("\nStage1 epochs:", STAGE1_DEFAULT_CONFIG.epochs)
print("Stage1 batch_size:", STAGE1_DEFAULT_CONFIG.batch_size)
print("Inference tile_size:", INFERENCE_DEFAULT_CONFIG.tile_size)

## 2. Preprocess (skip if `dataset_processed/` already exists)

```bash
python -m scripts.preprocess
```

## 3. Train Stage 1

```bash
python -m scripts.train_stage1
```

Override hyper-parameters from Python by constructing a `Stage1Config` and calling `main(cfg)` directly:

```python
from dataclasses import replace
from configs import STAGE1_DEFAULT_CONFIG
from symbiopan.training.stage1_trainer import main

cfg = replace(STAGE1_DEFAULT_CONFIG, epochs=2, batch_size=2)
result = main(cfg)
```

## 4. Run inference

```bash
python -m scripts.infer_wsi \
    --input  /path/to/wsi-tiles \
    --output /path/to/results \
    --cp     checkpoints/best_model.pth \
    --tta
```

## 5. Visualise a sample prediction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

OUT = Path("output/images/melanoma-tissue-mask-segmentation")
if not OUT.exists():
    print("No prediction found. Run inference first.")
else:
    import tifffile
    masks = sorted(OUT.glob("*.tif*"))
    if masks:
        mask = tifffile.imread(str(masks[0]))
        plt.figure(figsize=(6, 6))
        plt.imshow(mask, cmap="tab10", vmin=0, vmax=5)
        plt.title(f"Tissue prediction — {masks[0].name}")
        plt.axis("off")
        plt.show()